# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar el modelo que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [2]:
# Verificar que Colab asignó una GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga del modelo.

In [3]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [4]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.2


### **DESCARGAR EL MODELO**

Vamos a trabajar con **Llama 3.1 de 8B**. Guardamos el nombre en la variable `MODELO` para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [5]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)

MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


### **CLIENTE DE PYTHON**

In [6]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [7]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.message.content)

Zero-shot: La respuesta es: Mixto.


In [8]:
# Prompt de clasificación en modo few-shot

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: El sentimiento de la tercera reseña es Mixto, ya que la persona menciona una inconveniencia (el envío llegó tarde) pero también lo valora positivamente (el producto es excelente).


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [9]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

¡Vamos a resolver este problema paso a paso!

**Paso 1: Desea saber cuánto tiempo tarda el segundo tren en alcanzar al primero.**
Para resolver esto, primero debemos encontrar la distancia que el primer tren recorre en 2 horas.

**Paso 2: Encuentra la distancia recorrida por el primer tren en 2 horas.**
Distancia = Velocidad x Tiempo
= 80 km/h x 2 horas
= 160 km

**Paso 3: Desea saber la velocidad relativa entre el primer tren y el segundo tren.**
Dado que el segundo tren está saliendo de la misma ciudad, podemos asumir que está moviéndose en la misma dirección que el primer tren. Por lo tanto, la velocidad relativa es la diferencia entre sus velocidades.

**Paso 4: Encuentra la velocidad relativa entre el primer tren y el segundo tren.**
Velocidad relativa = Velocidad del segundo tren - Velocidad del primer tren
= 120 km/h - 80 km/h
= 40 km/h

**Paso 5: Desea saber el tiempo que tarda el segundo tren en cubrir la distancia que separa a los dos trenes.**
Dado que el primer tren ya ha v

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [10]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

Lo siento, pero no tengo información sobre un evento de hackathon específico realizado por DEV.F el 14 de agosto de 2026, ni sobre su resultado.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [25]:
# Instalar sentence-transformers

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [26]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [27]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [28]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

No, según nuestra política los productos en oferta no son elegibles para devolución.


## **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [29]:
# Verificar que Colab asignó una GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [30]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [31]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.2


In [32]:
# Descargar el modelo Llama e inicializar el cliente

MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


**2. Define tu base de conocimiento y genera embeddings:**

* Instala e importa la librería de  `sentence-transformers` junto con `numpy`.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings.

In [33]:
# Instalar e importar librerías

!pip install -q sentence-transformers numpy

In [34]:
# Definir la lista documentos y generar sus embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

# Modelo para generar embeddings multilingües
modelo_embeddings = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# Base de conocimiento de ejemplo
documentos = [
    "Las devoluciones se aceptan hasta 30 días después de "
    "la compra, con el producto en su empaque original.",

    "Los envíos internacionales no tienen devolución gratuita; "
    "el cliente cubre el costo del envío de regreso.",

    "Los productos en oferta o liquidación no son elegibles "
    "para devolución, solo para cambio de talla."
]

# Generar los embeddings de los documentos
embeddings_documentos = modelo_embeddings.encode(
    documentos,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Fragmentos cargados:", len(documentos))
print("Dimensiones de los embeddings:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fragmentos cargados: 3
Dimensiones de los embeddings: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [35]:
# Definir la función buscar_fragmento

import numpy as np

def buscar_fragmento(pregunta):
    """
    Busca el fragmento más relevante para una pregunta
    mediante similitud coseno.
    """

    # 1. Convertir la pregunta en un embedding
    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # 2. Calcular la similitud coseno con cada documento
    similitudes = np.dot(
        embeddings_documentos,
        embedding_pregunta
    )

    # 3. Identificar el fragmento con mayor similitud
    indice_relevante = int(np.argmax(similitudes))

    fragmento_relevante = documentos[indice_relevante]
    puntuacion = float(similitudes[indice_relevante])

    # 4. Regresar el fragmento más relevante
    return fragmento_relevante


# Prueba de funcionamiento
pregunta = "¿Puedo devolver un producto que compré en oferta?"

fragmento = buscar_fragmento(pregunta)

print("Pregunta:", pregunta)
print("\nFragmento más relevante:")
print(fragmento)


Pregunta: ¿Puedo devolver un producto que compré en oferta?

Fragmento más relevante:
Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [37]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag

# RETO 3. Generar una respuesta sin RAG y otra con RAG

pregunta = "¿Puedo devolver algo que compré en oferta?"

# A) Respuesta sin RAG
response_sin_rag = client.chat(
    model=MODELO,
    messages=[
        {
            "role": "user",
            "content": (
                "Responde de forma breve a esta pregunta. "
                "Si no tienes información suficiente, indica "
                "que no puedes confirmarlo.\nPregunta: "
                + pregunta
            )
        }
    ]
)

respuesta_sin_rag = response_sin_rag["message"]["content"]


# B) Respuesta con RAG
# Recuperar el fragmento relevante
fragmento = buscar_fragmento(pregunta)

prompt_rag = f"""Responde la pregunta usando ÚNICAMENTE
la política proporcionada.

Si la política no contiene la información necesaria,
dilo claramente. No inventes reglas.

Política: {fragmento}

Pregunta: {pregunta}

Da una respuesta breve."""

response_con_rag = client.chat(
    model=MODELO,
    messages=[
        {
            "role": "user",
            "content": prompt_rag
        }
    ]
)

respuesta_con_rag = response_con_rag["message"]["content"]


# Mostrar resultados
print("========== RESPUESTA SIN RAG ==========")
print(respuesta_sin_rag)

print("\n========== RESPUESTA CON RAG ==========")
print(respuesta_con_rag)

print("\n========== FRAGMENTO UTILIZADO ==========")
print(fragmento)

========== RESPUESTA SIN RAG ==========
Normalmente no se devuelven artículos de ofertas, ya que las condiciones de devolución suelen estar expresamente indicadas. Es mejor consultar el centro comercial o el establecimiento donde se adquirió el producto para obtener más información.

========== RESPUESTA CON RAG ==========
No.

========== FRAGMENTO UTILIZADO ==========
Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [38]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag

# RETO 4. Generar la respuesta con RAG

# 1. Definir la pregunta
pregunta = "¿Puedo devolver algo que compré en oferta?"

# 2. Recuperar el fragmento relevante
resultado = buscar_fragmento(pregunta)

# Si la función devuelve una tupla, extraemos el fragmento.
# Si devuelve solamente texto, usamos ese texto directamente.
if isinstance(resultado, tuple):
    fragmento = resultado[0]
else:
    fragmento = resultado

# 3. Preparar la pregunta junto con el fragmento recuperado
prompt_rag = f"""
Responde la siguiente pregunta utilizando únicamente
la información de la política proporcionada.

Si la política no contiene la información necesaria,
indícalo claramente. No inventes reglas.

Fragmento de la política:
{fragmento}

Pregunta:
{pregunta}

Responde de forma breve y clara.
"""

# 4. Enviar la información al modelo
response_con_rag = client.chat(
    model=MODELO,
    messages=[
        {
            "role": "user",
            "content": prompt_rag
        }
    ]
)

# 5. Guardar la respuesta en la variable solicitada
respuesta_con_rag = response_con_rag["message"]["content"]

# 6. Mostrar los resultados
print("PREGUNTA:")
print(pregunta)

print("\nFRAGMENTO RECUPERADO:")
print(fragmento)

print("\nRESPUESTA CON RAG:")
print(respuesta_con_rag)


PREGUNTA:
¿Puedo devolver algo que compré en oferta?

FRAGMENTO RECUPERADO:
Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.

RESPUESTA CON RAG:
No, los productos en oferta no son elegibles para devolución.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [39]:
# Mostrar ambas respuestas para comparar

# RETO 5. Comparar las respuestas y redactar una conclusión

print("========== PREGUNTA ANALIZADA ==========")
print(pregunta)

print("\n========== RESPUESTA SIN RAG ==========")
print(respuesta_sin_rag)

print("\n========== RESPUESTA CON RAG ==========")
print(respuesta_con_rag)

print("\n========== FRAGMENTO DE POLÍTICA RECUPERADO ==========")
print(fragmento)

print("\n========== CONCLUSIÓN ==========")

print(
    "La respuesta con RAG está más fundamentada cuando "
    "coincide con el fragmento recuperado, porque el modelo "
    "recibe información explícita de la política. La respuesta "
    "sin RAG no dispone de ese contexto y podría ser general "
    "o incorrecta. Es necesario verificar ambas respuestas "
    "contra el texto de la política antes de determinar cuál "
    "fue más precisa. RAG reduce el riesgo de alucinación, "
    "pero no garantiza su eliminación."
)

========== PREGUNTA ANALIZADA ==========
¿Puedo devolver algo que compré en oferta?

========== RESPUESTA SIN RAG ==========
Normalmente no se devuelven artículos de ofertas, ya que las condiciones de devolución suelen estar expresamente indicadas. Es mejor consultar el centro comercial o el establecimiento donde se adquirió el producto para obtener más información.

========== RESPUESTA CON RAG ==========
No, los productos en oferta no son elegibles para devolución.

========== FRAGMENTO DE POLÍTICA RECUPERADO ==========
Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.

========== CONCLUSIÓN ==========
La respuesta con RAG está más fundamentada cuando coincide con el fragmento recuperado, porque el modelo recibe información explícita de la política. La respuesta sin RAG no dispone de ese contexto y podría ser general o incorrecta. Es necesario verificar ambas respuestas contra el texto de la política antes de determinar cuál fue más pr